# Análisis exploratorio de anuncios de vehículos usados

**Sprint 7 — Herramientas de desarrollo de software**

Este notebook explora el conjunto de datos `vehicles_us.csv` para fundamentar el dashboard web. Se priorizan la calidad de los datos, las distribuciones y la relación descriptiva entre precio y kilometraje.

## 1. Carga y reproducibilidad

Se utiliza una ruta relativa desde el directorio `notebooks` para facilitar la ejecución dentro del repositorio.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

car_data = pd.read_csv(Path('../vehicles_us.csv'))
car_data.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [2]:
print(f'Filas: {car_data.shape[0]:,}')
print(f'Columnas: {car_data.shape[1]}')
car_data.info()

Filas: 51,525
Columnas: 13
<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


## 2. Calidad de los datos

Se comprueban duplicados y valores ausentes. No se imputan indiscriminadamente atributos desconocidos: hacerlo podría distorsionar las distribuciones. En cada visualización se excluyen solo las filas donde falta la variable necesaria.

In [3]:
missing = (car_data.isna().sum().to_frame(name='valores_ausentes')
           .assign(porcentaje=lambda x: (100 * x['valores_ausentes'] / len(car_data)).round(2))
           .sort_values('valores_ausentes', ascending=False))
print(f'Filas duplicadas exactas: {car_data.duplicated().sum()}')
missing

Filas duplicadas exactas: 0


,valores_ausentes,porcentaje
is_4wd,25953,50.37
paint_color,9267,17.99
odometer,7892,15.32
cylinders,5260,10.21
model_year,3619,7.02
condition,0,0.00
model,0,0.00
price,0,0.00
fuel,0,0.00
type,0,0.00


El campo `is_4wd` presenta valores ausentes. No se interpreta automáticamente que todos ellos signifiquen ausencia de tracción 4x4, porque la fuente no documenta ese supuesto.

## 3. Estadísticas descriptivas

Además de la media se utiliza la mediana, una medida más robusta ante precios anunciados excepcionalmente elevados.

In [4]:
car_data[['price', 'odometer', 'model_year', 'days_listed']].describe()

,price,odometer,model_year,days_listed
count,51525.000000,43633.000000,47906.000000,51525.00000
mean,12132.464920,115553.461738,2009.750470,39.55476
std,10040.803015,65094.611341,6.282065,28.20427
min,1.000000,0.000000,1908.000000,0.00000
25%,5000.000000,70000.000000,2006.000000,19.00000
50%,9000.000000,113000.000000,2011.000000,33.00000
75%,16839.000000,155000.000000,2014.000000,53.00000
max,375000.000000,990000.000000,2019.000000,271.00000


In [5]:
print('Precio mediano (USD):', car_data['price'].median())
print('Kilometraje mediano (millas):', car_data['odometer'].median())
print('Tipos de vehículo más frecuentes:')
print(car_data['type'].value_counts().head(5))

Precio mediano (USD): 9000.0
Kilometraje mediano (millas): 113000.0
Tipos de vehículo más frecuentes:
type
SUV       12405
truck     12353
sedan     12154
pickup     6988
coupe      2303
Name: count, dtype: int64


## 4. Distribución del kilometraje

El histograma muestra dónde se concentran los vehículos anunciados y deja ver los valores extremos. Se eliminan únicamente los registros que no tienen kilometraje.

In [6]:
odo_data = car_data.dropna(subset=['odometer'])
fig_hist = px.histogram(odo_data, x='odometer', nbins=50,
    title='Distribución del kilometraje de vehículos anunciados',
    labels={'odometer': 'Kilometraje (millas)'})
fig_hist.update_layout(yaxis_title='Número de anuncios')
fig_hist.show()

## 5. Relación entre kilometraje y precio

El diagrama de dispersión permite examinar la asociación entre variables numéricas, sin demostrar causalidad. Para evitar sobrecargar el navegador, se toma una muestra aleatoria reproducible de hasta 5.000 anuncios completos.

In [7]:
scatter_data = car_data.dropna(subset=['price', 'odometer']).copy()
if len(scatter_data) > 5_000:
    scatter_data = scatter_data.sample(n=5_000, random_state=42)
fig_scatter = px.scatter(scatter_data, x='odometer', y='price', color='condition',
    opacity=0.5, render_mode='webgl', title='Precio anunciado frente a kilometraje',
    labels={'odometer': 'Kilometraje (millas)', 'price': 'Precio anunciado (USD)',
            'condition': 'Condición'})
fig_scatter.show()

## 6. Comparación por tipo de vehículo

Se analizan conteos y medianas. Estas cifras descriptivas no controlan por marca, año ni estado; por tanto, no prueban que el tipo del vehículo cause las diferencias de precio.

In [8]:
summary_by_type = (car_data.groupby('type', dropna=False)
    .agg(anuncios=('price', 'size'),
         precio_mediano_usd=('price', 'median'),
         kilometraje_mediano_millas=('odometer', 'median'))
    .sort_values('anuncios', ascending=False))
summary_by_type.head(10)

,anuncios,precio_mediano_usd,kilometraje_mediano_millas
type,,,
SUV,12405,8900.0,118865.0
truck,12353,14995.0,120000.0
sedan,12154,5995.0,105000.0
pickup,6988,14200.0,118622.0
coupe,2303,12950.0,63070.0
wagon,1541,7900.0,121467.0
mini-van,1161,6495.0,126350.0
hatchback,1047,5980.0,111507.0
van,633,7990.0,123362.0


## 7. Conclusiones

- El dataset incluye **51.525 anuncios** y **13 variables**.
- El precio mediano anunciado es **9.000 USD** y el kilometraje mediano registrado es **113.000 millas**.
- Los tipos más comunes son **SUV (12.405)**, **truck (12.353)** y **sedan (12.154)**.
- Faltan odómetros en **7.892** anuncios y años de modelo en **3.619**.
- Los precios son valores anunciados, no ventas confirmadas. Las relaciones visuales no establecen causalidad y los valores extremos pueden alterar las medias.

**Decisión para la aplicación:** incluir histograma de kilometraje, dispersión entre precio y kilometraje, filtro por tipo de vehículo y métricas descriptivas; son funcionalidades pertinentes a la consigna.